In [18]:
import string
import pandas as pd


class BowVector:
	def __init__(self, texts: list[str]):
		self.texts = texts
		self.words = self.__getWordList()
		self.vector = self.createVector(self.texts)

	def cleanTexts(self, texts: list[str]):
		return [
			text.lower().translate(str.maketrans("", "", string.punctuation))
			for text in texts
		]

	def __getWordList(self):
		# Собираем все строки в одну
		combined_string = " ".join(self.cleanTexts(self.texts))
		# Создаем сет уникальный слов , а также избавляемся от коротких цифр и тд
		return [word for word in set(combined_string.split()) if len(word) > 1]


	def createVector(self, texts):
		cleanedTexts = self.cleanTexts(texts)
		word_set = self.__getWordList()
		# Фильтруем наш входящий document_list, чтобы он тоже содержал слова более длинны 1
		filtered_docs = []
		for document in cleanedTexts:
			filtered_docs.append(
				" ".join([word for word in document.split() if len(word) > 1])
			)

		doc_word_count = []
		for document in filtered_docs:
			# split строки по документу, то есть текст в массив и здесь же цикл посчитать сколько по нашему сету слов совпадений
			doc_word_count.append([document.split().count(word) for word in word_set])

		return {doc: vector for doc, vector in zip(filtered_docs, doc_word_count)}

	def searchSimilarDocument(self, search_vector):
		# Search in Bow vector (for next simple lexical search)
		vectors = list(self.vector.values())
		# Match score
		score = {}
		for i,doc in enumerate(search_vector):
			for j,vector in enumerate(vectors):
				if vectors[j][i] == doc and vectors[j][i] > 0:
					score.setdefault(j, {})
					score[j][i] = score[j].get(i, 0) + 1
		# Filter by best score
		scoreFiltered = {
			"best": 0,
			"doc": 0
		}
		for doc in score:
			if len(score[doc]) > scoreFiltered["best"]:
				scoreFiltered["best"] = len(score[doc])
				scoreFiltered["doc"] = doc
		# Get word my best score
		findedWords = []
		for i in score[scoreFiltered["doc"]]:
			findedWords.append(Vector.words[i])
		return findedWords



# Data set
news_headlines = [
	"Stocks Slide After Jobless Claims Rise",
	"Jobless Claims Rose Last Week, Still Historically Low",
	"Stocks Extend Losses, Reversing Early-Week Gains",
	"Jobless Claims Are Expected to Rise",
	"More Americans apply for jobless benefits last week",
	"NFL Week 4 Offense Rankings | NFL News, Rankings and Statistics",
	"NFL rankings, figuring out the Ravens offense, Kenny Pickett gets the nod",
	"Week 4's NFL Team of the Week - Offense",
	"2022 NFL offense rankings",
	"How Will Every NFL Offense Perform in 2022?",
]


Vector = BowVector(news_headlines)

df = pd.DataFrame(
	data=Vector.vector.values(),
	columns=Vector.words,
	index=Vector.vector.keys(),
)
df.head()


,news,claims,extend,statistics,earlyweek,historically,stocks,team,how,in,...,apply,benefits,last,perform,reversing,out,still,rankings,rise,expected
stocks slide after jobless claims rise,0,1,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
jobless claims rose last week still historically low,0,1,0,0,0,1,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
stocks extend losses reversing earlyweek gains,0,0,1,0,1,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,0
jobless claims are expected to rise,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
more americans apply for jobless benefits last week,0,0,0,0,0,0,0,0,0,0,...,1,1,1,0,0,0,0,0,0,0


In [19]:


# Search in Bow vector (for next simple lexical search)
search_vector = Vector.vector['stocks extend losses reversing earlyweek gains']
words = Vector.searchSimilarDocument(search_vector)
print("Getting words from exist vector:",words)

# Creating a new vector and try search a words
new_vector = Vector.createVector(["A new text Jobless"])['new text jobless']
new_w_words = Vector.searchSimilarDocument(new_vector)
print("Getting words from a new vector:", new_w_words)

Getting words from exist vector: ['extend', 'earlyweek', 'stocks', 'gains', 'losses', 'reversing']
Getting words from a new vector: ['jobless']
